<a href="https://colab.research.google.com/github/mislam2106/GitHub-Training/blob/main/%F0%9F%8D%81Project4_Paper_Company_Sales_Team_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project4: The Beaver's Choice Paper Company Sales Team

0.0 Original project work <starter.py>


Don't run
```
# Use as a reference
#==================================
import pandas as pd
import numpy as np
import os
import time
import dotenv
import ast
from sqlalchemy.sql import text
from datetime import datetime, timedelta
from typing import Dict, List, Union
from sqlalchemy import create_engine, Engine

# Create an SQLite database
db_engine = create_engine("sqlite:///munder_difflin.db")

# List containing the different kinds of papers
paper_supplies = [
    # Paper Types (priced per sheet unless specified)
    {"item_name": "A4 paper",                         "category": "paper",        "unit_price": 0.05},
    {"item_name": "Letter-sized paper",              "category": "paper",        "unit_price": 0.06},
    {"item_name": "Cardstock",                        "category": "paper",        "unit_price": 0.15},
    {"item_name": "Colored paper",                    "category": "paper",        "unit_price": 0.10},
    {"item_name": "Glossy paper",                     "category": "paper",        "unit_price": 0.20},
    {"item_name": "Matte paper",                      "category": "paper",        "unit_price": 0.18},
    {"item_name": "Recycled paper",                   "category": "paper",        "unit_price": 0.08},
    {"item_name": "Eco-friendly paper",               "category": "paper",        "unit_price": 0.12},
    {"item_name": "Poster paper",                     "category": "paper",        "unit_price": 0.25},
    {"item_name": "Banner paper",                     "category": "paper",        "unit_price": 0.30},
    {"item_name": "Kraft paper",                      "category": "paper",        "unit_price": 0.10},
    {"item_name": "Construction paper",               "category": "paper",        "unit_price": 0.07},
    {"item_name": "Wrapping paper",                   "category": "paper",        "unit_price": 0.15},
    {"item_name": "Glitter paper",                    "category": "paper",        "unit_price": 0.22},
    {"item_name": "Decorative paper",                 "category": "paper",        "unit_price": 0.18},
    {"item_name": "Letterhead paper",                 "category": "paper",        "unit_price": 0.12},
    {"item_name": "Legal-size paper",                 "category": "paper",        "unit_price": 0.08},
    {"item_name": "Crepe paper",                      "category": "paper",        "unit_price": 0.05},
    {"item_name": "Photo paper",                      "category": "paper",        "unit_price": 0.25},
    {"item_name": "Uncoated paper",                   "category": "paper",        "unit_price": 0.06},
    {"item_name": "Butcher paper",                    "category": "paper",        "unit_price": 0.10},
    {"item_name": "Heavyweight paper",                "category": "paper",        "unit_price": 0.20},
    {"item_name": "Standard copy paper",              "category": "paper",        "unit_price": 0.04},
    {"item_name": "Bright-colored paper",             "category": "paper",        "unit_price": 0.12},
    {"item_name": "Patterned paper",                  "category": "paper",        "unit_price": 0.15},

    # Product Types (priced per unit)
    {"item_name": "Paper plates",                     "category": "product",      "unit_price": 0.10},  # per plate
    {"item_name": "Paper cups",                       "category": "product",      "unit_price": 0.08},  # per cup
    {"item_name": "Paper napkins",                    "category": "product",      "unit_price": 0.02},  # per napkin
    {"item_name": "Disposable cups",                  "category": "product",      "unit_price": 0.10},  # per cup
    {"item_name": "Table covers",                     "category": "product",      "unit_price": 1.50},  # per cover
    {"item_name": "Envelopes",                        "category": "product",      "unit_price": 0.05},  # per envelope
    {"item_name": "Sticky notes",                     "category": "product",      "unit_price": 0.03},  # per sheet
    {"item_name": "Notepads",                         "category": "product",      "unit_price": 2.00},  # per pad
    {"item_name": "Invitation cards",                 "category": "product",      "unit_price": 0.50},  # per card
    {"item_name": "Flyers",                           "category": "product",      "unit_price": 0.15},  # per flyer
    {"item_name": "Party streamers",                  "category": "product",      "unit_price": 0.05},  # per roll
    {"item_name": "Decorative adhesive tape (washi tape)", "category": "product", "unit_price": 0.20},  # per roll
    {"item_name": "Paper party bags",                 "category": "product",      "unit_price": 0.25},  # per bag
    {"item_name": "Name tags with lanyards",          "category": "product",      "unit_price": 0.75},  # per tag
    {"item_name": "Presentation folders",             "category": "product",      "unit_price": 0.50},  # per folder

    # Large-format items (priced per unit)
    {"item_name": "Large poster paper (24x36 inches)", "category": "large_format", "unit_price": 1.00},
    {"item_name": "Rolls of banner paper (36-inch width)", "category": "large_format", "unit_price": 2.50},

    # Specialty papers
    {"item_name": "100 lb cover stock",               "category": "specialty",    "unit_price": 0.50},
    {"item_name": "80 lb text paper",                 "category": "specialty",    "unit_price": 0.40},
    {"item_name": "250 gsm cardstock",                "category": "specialty",    "unit_price": 0.30},
    {"item_name": "220 gsm poster paper",             "category": "specialty",    "unit_price": 0.35},
]

# Given below are some utility functions you can use to implement your multi-agent system

def generate_sample_inventory(paper_supplies: list, coverage: float = 0.4, seed: int = 137) -> pd.DataFrame:
    """
    Generate inventory for exactly a specified percentage of items from the full paper supply list.

    This function randomly selects exactly `coverage` × N items from the `paper_supplies` list,
    and assigns each selected item:
    - a random stock quantity between 200 and 800,
    - a minimum stock level between 50 and 150.

    The random seed ensures reproducibility of selection and stock levels.

    Args:
        paper_supplies (list): A list of dictionaries, each representing a paper item with
                               keys 'item_name', 'category', and 'unit_price'.
        coverage (float, optional): Fraction of items to include in the inventory (default is 0.4, or 40%).
        seed (int, optional): Random seed for reproducibility (default is 137).

    Returns:
        pd.DataFrame: A DataFrame with the selected items and assigned inventory values, including:
                      - item_name
                      - category
                      - unit_price
                      - current_stock
                      - min_stock_level
    """
    # Ensure reproducible random output
    np.random.seed(seed)

    # Calculate number of items to include based on coverage
    num_items = int(len(paper_supplies) * coverage)

    # Randomly select item indices without replacement
    selected_indices = np.random.choice(
        range(len(paper_supplies)),
        size=num_items,
        replace=False
    )

    # Extract selected items from paper_supplies list
    selected_items = [paper_supplies[i] for i in selected_indices]

    # Construct inventory records
    inventory = []
    for item in selected_items:
        inventory.append({
            "item_name": item["item_name"],
            "category": item["category"],
            "unit_price": item["unit_price"],
            "current_stock": np.random.randint(200, 800),  # Realistic stock range
            "min_stock_level": np.random.randint(50, 150)  # Reasonable threshold for reordering
        })

    # Return inventory as a pandas DataFrame
    return pd.DataFrame(inventory)

def init_database(db_engine: Engine, seed: int = 137) -> Engine:    
    """
    Set up the Munder Difflin database with all required tables and initial records.

    This function performs the following tasks:
    - Creates the 'transactions' table for logging stock orders and sales
    - Loads customer inquiries from 'quote_requests.csv' into a 'quote_requests' table
    - Loads previous quotes from 'quotes.csv' into a 'quotes' table, extracting useful metadata
    - Generates a random subset of paper inventory using `generate_sample_inventory`
    - Inserts initial financial records including available cash and starting stock levels

    Args:
        db_engine (Engine): A SQLAlchemy engine connected to the SQLite database.
        seed (int, optional): A random seed used to control reproducibility of inventory stock levels.
                              Default is 137.

    Returns:
        Engine: The same SQLAlchemy engine, after initializing all necessary tables and records.

    Raises:
        Exception: If an error occurs during setup, the exception is printed and raised.
    """
    try:
        # ----------------------------
        # 1. Create an empty 'transactions' table schema
        # ----------------------------
        transactions_schema = pd.DataFrame({
            "id": [],
            "item_name": [],
            "transaction_type": [],  # 'stock_orders' or 'sales'
            "units": [],             # Quantity involved
            "price": [],             # Total price for the transaction
            "transaction_date": [],  # ISO-formatted date
        })
        transactions_schema.to_sql("transactions", db_engine, if_exists="replace", index=False)

        # Set a consistent starting date
        initial_date = datetime(2025, 1, 1).isoformat()

        # ----------------------------
        # 2. Load and initialize 'quote_requests' table
        # ----------------------------
        quote_requests_df = pd.read_csv("quote_requests.csv")
        quote_requests_df["id"] = range(1, len(quote_requests_df) + 1)
        quote_requests_df.to_sql("quote_requests", db_engine, if_exists="replace", index=False)

        # ----------------------------
        # 3. Load and transform 'quotes' table
        # ----------------------------
        quotes_df = pd.read_csv("quotes.csv")
        quotes_df["request_id"] = range(1, len(quotes_df) + 1)
        quotes_df["order_date"] = initial_date

        # Unpack metadata fields (job_type, order_size, event_type) if present
        if "request_metadata" in quotes_df.columns:
            quotes_df["request_metadata"] = quotes_df["request_metadata"].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            quotes_df["job_type"] = quotes_df["request_metadata"].apply(lambda x: x.get("job_type", ""))
            quotes_df["order_size"] = quotes_df["request_metadata"].apply(lambda x: x.get("order_size", ""))
            quotes_df["event_type"] = quotes_df["request_metadata"].apply(lambda x: x.get("event_type", ""))

        # Retain only relevant columns
        quotes_df = quotes_df[[
            "request_id",
            "total_amount",
            "quote_explanation",
            "order_date",
            "job_type",
            "order_size",
            "event_type"
        ]]
        quotes_df.to_sql("quotes", db_engine, if_exists="replace", index=False)

        # ----------------------------
        # 4. Generate inventory and seed stock
        # ----------------------------
        inventory_df = generate_sample_inventory(paper_supplies, seed=seed)

        # Seed initial transactions
        initial_transactions = []

        # Add a starting cash balance via a dummy sales transaction
        initial_transactions.append({
            "item_name": None,
            "transaction_type": "sales",
            "units": None,
            "price": 50000.0,
            "transaction_date": initial_date,
        })

        # Add one stock order transaction per inventory item
        for _, item in inventory_df.iterrows():
            initial_transactions.append({
                "item_name": item["item_name"],
                "transaction_type": "stock_orders",
                "units": item["current_stock"],
                "price": item["current_stock"] * item["unit_price"],
                "transaction_date": initial_date,
            })

        # Commit transactions to database
        pd.DataFrame(initial_transactions).to_sql("transactions", db_engine, if_exists="append", index=False)

        # Save the inventory reference table
        inventory_df.to_sql("inventory", db_engine, if_exists="replace", index=False)

        return db_engine

    except Exception as e:
        print(f"Error initializing database: {e}")
        raise

def create_transaction(
    item_name: str,
    transaction_type: str,
    quantity: int,
    price: float,
    date: Union[str, datetime],
) -> int:
    """
    This function records a transaction of type 'stock_orders' or 'sales' with a specified
    item name, quantity, total price, and transaction date into the 'transactions' table of the database.

    Args:
        item_name (str): The name of the item involved in the transaction.
        transaction_type (str): Either 'stock_orders' or 'sales'.
        quantity (int): Number of units involved in the transaction.
        price (float): Total price of the transaction.
        date (str or datetime): Date of the transaction in ISO 8601 format.

    Returns:
        int: The ID of the newly inserted transaction.

    Raises:
        ValueError: If `transaction_type` is not 'stock_orders' or 'sales'.
        Exception: For other database or execution errors.
    """
    try:
        # Convert datetime to ISO string if necessary
        date_str = date.isoformat() if isinstance(date, datetime) else date

        # Validate transaction type
        if transaction_type not in {"stock_orders", "sales"}:
            raise ValueError("Transaction type must be 'stock_orders' or 'sales'")

        # Prepare transaction record as a single-row DataFrame
        transaction = pd.DataFrame([{
            "item_name": item_name,
            "transaction_type": transaction_type,
            "units": quantity,
            "price": price,
            "transaction_date": date_str,
        }])

        # Insert the record into the database
        transaction.to_sql("transactions", db_engine, if_exists="append", index=False)

        # Fetch and return the ID of the inserted row
        result = pd.read_sql("SELECT last_insert_rowid() as id", db_engine)
        return int(result.iloc[0]["id"])

    except Exception as e:
        print(f"Error creating transaction: {e}")
        raise

def get_all_inventory(as_of_date: str) -> Dict[str, int]:
    """
    Retrieve a snapshot of available inventory as of a specific date.

    This function calculates the net quantity of each item by summing
    all stock orders and subtracting all sales up to and including the given date.

    Only items with positive stock are included in the result.

    Args:
        as_of_date (str): ISO-formatted date string (YYYY-MM-DD) representing the inventory cutoff.

    Returns:
        Dict[str, int]: A dictionary mapping item names to their current stock levels.
    """
    # SQL query to compute stock levels per item as of the given date
    query = """
        SELECT
            item_name,
            SUM(CASE
                WHEN transaction_type = 'stock_orders' THEN units
                WHEN transaction_type = 'sales' THEN -units
                ELSE 0
            END) as stock
        FROM transactions
        WHERE item_name IS NOT NULL
        AND transaction_date <= :as_of_date
        GROUP BY item_name
        HAVING stock > 0
    """

    # Execute the query with the date parameter
    result = pd.read_sql(query, db_engine, params={"as_of_date": as_of_date})

    # Convert the result into a dictionary {item_name: stock}
    return dict(zip(result["item_name"], result["stock"]))

def get_stock_level(item_name: str, as_of_date: Union[str, datetime]) -> pd.DataFrame:
    """
    Retrieve the stock level of a specific item as of a given date.

    This function calculates the net stock by summing all 'stock_orders' and
    subtracting all 'sales' transactions for the specified item up to the given date.

    Args:
        item_name (str): The name of the item to look up.
        as_of_date (str or datetime): The cutoff date (inclusive) for calculating stock.

    Returns:
        pd.DataFrame: A single-row DataFrame with columns 'item_name' and 'current_stock'.
    """
    # Convert date to ISO string format if it's a datetime object
    if isinstance(as_of_date, datetime):
        as_of_date = as_of_date.isoformat()

    # SQL query to compute net stock level for the item
    stock_query = """
        SELECT
            item_name,
            COALESCE(SUM(CASE
                WHEN transaction_type = 'stock_orders' THEN units
                WHEN transaction_type = 'sales' THEN -units
                ELSE 0
            END), 0) AS current_stock
        FROM transactions
        WHERE item_name = :item_name
        AND transaction_date <= :as_of_date
    """

    # Execute query and return result as a DataFrame
    return pd.read_sql(
        stock_query,
        db_engine,
        params={"item_name": item_name, "as_of_date": as_of_date},
    )

def get_supplier_delivery_date(input_date_str: str, quantity: int) -> str:
    """
    Estimate the supplier delivery date based on the requested order quantity and a starting date.

    Delivery lead time increases with order size:
        - ≤10 units: same day
        - 11–100 units: 1 day
        - 101–1000 units: 4 days
        - >1000 units: 7 days

    Args:
        input_date_str (str): The starting date in ISO format (YYYY-MM-DD).
        quantity (int): The number of units in the order.

    Returns:
        str: Estimated delivery date in ISO format (YYYY-MM-DD).
    """
    # Debug log (comment out in production if needed)
    print(f"FUNC (get_supplier_delivery_date): Calculating for qty {quantity} from date string '{input_date_str}'")

    # Attempt to parse the input date
    try:
        input_date_dt = datetime.fromisoformat(input_date_str.split("T")[0])
    except (ValueError, TypeError):
        # Fallback to current date on format error
        print(f"WARN (get_supplier_delivery_date): Invalid date format '{input_date_str}', using today as base.")
        input_date_dt = datetime.now()

    # Determine delivery delay based on quantity
    if quantity <= 10:
        days = 0
    elif quantity <= 100:
        days = 1
    elif quantity <= 1000:
        days = 4
    else:
        days = 7

    # Add delivery days to the starting date
    delivery_date_dt = input_date_dt + timedelta(days=days)

    # Return formatted delivery date
    return delivery_date_dt.strftime("%Y-%m-%d")

def get_cash_balance(as_of_date: Union[str, datetime]) -> float:
    """
    Calculate the current cash balance as of a specified date.

    The balance is computed by subtracting total stock purchase costs ('stock_orders')
    from total revenue ('sales') recorded in the transactions table up to the given date.

    Args:
        as_of_date (str or datetime): The cutoff date (inclusive) in ISO format or as a datetime object.

    Returns:
        float: Net cash balance as of the given date. Returns 0.0 if no transactions exist or an error occurs.
    """
    try:
        # Convert date to ISO format if it's a datetime object
        if isinstance(as_of_date, datetime):
            as_of_date = as_of_date.isoformat()

        # Query all transactions on or before the specified date
        transactions = pd.read_sql(
            "SELECT * FROM transactions WHERE transaction_date <= :as_of_date",
            db_engine,
            params={"as_of_date": as_of_date},
        )

        # Compute the difference between sales and stock purchases
        if not transactions.empty:
            total_sales = transactions.loc[transactions["transaction_type"] == "sales", "price"].sum()
            total_purchases = transactions.loc[transactions["transaction_type"] == "stock_orders", "price"].sum()
            return float(total_sales - total_purchases)

        return 0.0

    except Exception as e:
        print(f"Error getting cash balance: {e}")
        return 0.0


def generate_financial_report(as_of_date: Union[str, datetime]) -> Dict:
    """
    Generate a complete financial report for the company as of a specific date.

    This includes:
    - Cash balance
    - Inventory valuation
    - Combined asset total
    - Itemized inventory breakdown
    - Top 5 best-selling products

    Args:
        as_of_date (str or datetime): The date (inclusive) for which to generate the report.

    Returns:
        Dict: A dictionary containing the financial report fields:
            - 'as_of_date': The date of the report
            - 'cash_balance': Total cash available
            - 'inventory_value': Total value of inventory
            - 'total_assets': Combined cash and inventory value
            - 'inventory_summary': List of items with stock and valuation details
            - 'top_selling_products': List of top 5 products by revenue
    """
    # Normalize date input
    if isinstance(as_of_date, datetime):
        as_of_date = as_of_date.isoformat()

    # Get current cash balance
    cash = get_cash_balance(as_of_date)

    # Get current inventory snapshot
    inventory_df = pd.read_sql("SELECT * FROM inventory", db_engine)
    inventory_value = 0.0
    inventory_summary = []

    # Compute total inventory value and summary by item
    for _, item in inventory_df.iterrows():
        stock_info = get_stock_level(item["item_name"], as_of_date)
        stock = stock_info["current_stock"].iloc[0]
        item_value = stock * item["unit_price"]
        inventory_value += item_value

        inventory_summary.append({
            "item_name": item["item_name"],
            "stock": stock,
            "unit_price": item["unit_price"],
            "value": item_value,
        })

    # Identify top-selling products by revenue
    top_sales_query = """
        SELECT item_name, SUM(units) as total_units, SUM(price) as total_revenue
        FROM transactions
        WHERE transaction_type = 'sales' AND transaction_date <= :date
        GROUP BY item_name
        ORDER BY total_revenue DESC
        LIMIT 5
    """
    top_sales = pd.read_sql(top_sales_query, db_engine, params={"date": as_of_date})
    top_selling_products = top_sales.to_dict(orient="records")

    return {
        "as_of_date": as_of_date,
        "cash_balance": cash,
        "inventory_value": inventory_value,
        "total_assets": cash + inventory_value,
        "inventory_summary": inventory_summary,
        "top_selling_products": top_selling_products,
    }


def search_quote_history(search_terms: List[str], limit: int = 5) -> List[Dict]:
    """
    Retrieve a list of historical quotes that match any of the provided search terms.

    The function searches both the original customer request (from `quote_requests`) and
    the explanation for the quote (from `quotes`) for each keyword. Results are sorted by
    most recent order date and limited by the `limit` parameter.

    Args:
        search_terms (List[str]): List of terms to match against customer requests and explanations.
        limit (int, optional): Maximum number of quote records to return. Default is 5.

    Returns:
        List[Dict]: A list of matching quotes, each represented as a dictionary with fields:
            - original_request
            - total_amount
            - quote_explanation
            - job_type
            - order_size
            - event_type
            - order_date
    """
    conditions = []
    params = {}

    # Build SQL WHERE clause using LIKE filters for each search term
    for i, term in enumerate(search_terms):
        param_name = f"term_{i}"
        conditions.append(
            f"(LOWER(qr.response) LIKE :{param_name} OR "
            f"LOWER(q.quote_explanation) LIKE :{param_name})"
        )
        params[param_name] = f"%{term.lower()}%"

    # Combine conditions; fallback to always-true if no terms provided
    where_clause = " AND ".join(conditions) if conditions else "1=1"

    # Final SQL query to join quotes with quote_requests
    query = f"""
        SELECT
            qr.response AS original_request,
            q.total_amount,
            q.quote_explanation,
            q.job_type,
            q.order_size,
            q.event_type,
            q.order_date
        FROM quotes q
        JOIN quote_requests qr ON q.request_id = qr.id
        WHERE {where_clause}
        ORDER BY q.order_date DESC
        LIMIT {limit}
    """

    # Execute parameterized query
    with db_engine.connect() as conn:
        result = conn.execute(text(query), params)
        return [dict(row._mapping) for row in result]

########################
########################
########################
# YOUR MULTI AGENT STARTS HERE
########################
########################
########################


# Set up and load your env parameters and instantiate your model.


"""Set up tools for your agents to use, these should be methods that combine the database functions above
 and apply criteria to them to ensure that the flow of the system is correct."""


# Tools for inventory agent


# Tools for quoting agent


# Tools for ordering agent


# Set up your agents and create an orchestration agent that will manage them.


# Run your test scenarios by writing them here. Make sure to keep track of them.

def run_test_scenarios():
    
    print("Initializing Database...")
    init_database()
    try:
        quote_requests_sample = pd.read_csv("quote_requests_sample.csv")
        quote_requests_sample["request_date"] = pd.to_datetime(
            quote_requests_sample["request_date"], format="%m/%d/%y", errors="coerce"
        )
        quote_requests_sample.dropna(subset=["request_date"], inplace=True)
        quote_requests_sample = quote_requests_sample.sort_values("request_date")
    except Exception as e:
        print(f"FATAL: Error loading test data: {e}")
        return

    # Get initial state
    initial_date = quote_requests_sample["request_date"].min().strftime("%Y-%m-%d")
    report = generate_financial_report(initial_date)
    current_cash = report["cash_balance"]
    current_inventory = report["inventory_value"]

    ############
    ############
    ############
    # INITIALIZE YOUR MULTI AGENT SYSTEM HERE
    ############
    ############
    ############

    results = []
    for idx, row in quote_requests_sample.iterrows():
        request_date = row["request_date"].strftime("%Y-%m-%d")

        print(f"\n=== Request {idx+1} ===")
        print(f"Context: {row['job']} organizing {row['event']}")
        print(f"Request Date: {request_date}")
        print(f"Cash Balance: ${current_cash:.2f}")
        print(f"Inventory Value: ${current_inventory:.2f}")

        # Process request
        request_with_date = f"{row['request']} (Date of request: {request_date})"

        ############
        ############
        ############
        # USE YOUR MULTI AGENT SYSTEM TO HANDLE THE REQUEST
        ############
        ############
        ############

        # response = call_your_multi_agent_system(request_with_date)

        # Update state
        report = generate_financial_report(request_date)
        current_cash = report["cash_balance"]
        current_inventory = report["inventory_value"]

        print(f"Response: {response}")
        print(f"Updated Cash: ${current_cash:.2f}")
        print(f"Updated Inventory: ${current_inventory:.2f}")

        results.append(
            {
                "request_id": idx + 1,
                "request_date": request_date,
                "cash_balance": current_cash,
                "inventory_value": current_inventory,
                "response": response,
            }
        )

        time.sleep(1)

    # Final report
    final_date = quote_requests_sample["request_date"].max().strftime("%Y-%m-%d")
    final_report = generate_financial_report(final_date)
    print("\n===== FINAL FINANCIAL REPORT =====")
    print(f"Final Cash: ${final_report['cash_balance']:.2f}")
    print(f"Final Inventory: ${final_report['inventory_value']:.2f}")

    # Save results
    pd.DataFrame(results).to_csv("test_results.csv", index=False)
    return results


if __name__ == "__main__":
    results = run_test_scenarios()

```



# Project Start Here
=========================================================
1. Local setup

In [6]:
import openai
# To securely store your API key, use Colab's UserData secrets management
from google.colab import userdata

# Load your personal OpenAI API Key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client directly with your personal API key
# Remove the 'base_url' if you are connecting to the official OpenAI API
openai_client = openai.OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI client re-initialized with personal API key.")

OpenAI client re-initialized with personal API key.


## Multi-Agent System Flowchart

```mermaid
graph TD
    User_Request[User Request] --> Orchestrator_LLM;

    subgraph Orchestrator
        Orchestrator_LLM{"<b> Orchestrator (LLM)<b> - <br>Tool Selection & Response Generation"}
    end

    subgraph Inventory_Agent[<b>Inventory Agent - <i> manages & tracks stock levels]
        get_item_stock["get_item_stock - Checks current stock levels via get_stock_level()"]
        get_full_inventory_summary["get_full_inventory_summary - Provides full inventory summary via get_all_inventory()"]
    end

    subgraph Quoting_Agent[<b>Quoting Agent - <i> quote with price, discount, and reference basis]
        find_previous_quotes["find_previous_quotes - Searches historical quotes via search_quote_history()"]
        generate_price_quote["generate_price_quote - Calculates discounted quote via get_item_unit_price()"]
    end

    subgraph Ordering_Agent[<b>Ordering Agent - <i> order confirmation with price, quantity, and delivery date]
        get_item_unit_price["get_item_unit_price - Retrieves unit price from paper_supplies list (fuzzy match)"]
        place_order["place_order - Records sales/stock orders via create_transaction()"]
        estimate_delivery_date["estimate_delivery_date - Estimates delivery lead time via get_supplier_delivery_date()"]
    end

    Database[("Database - (db_engine)" <br>munder_difflin.db)];
    Date_Calculation_Logic((Date Calculation Logic));
    Product_Catalog[(Product Catalog - paper_supplies list)];

    
    get_item_stock -- Query --> Database;
    get_full_inventory_summary -- Query --> Database;
    find_previous_quotes -- Query --> Database;
    place_order -- Write --> Database;
    estimate_delivery_date -- Calculate --> Date_Calculation_Logic;

    
    generate_price_quote -- calls --> get_item_unit_price;
    get_item_unit_price -- Lookup --> Product_Catalog;


    
    Orchestrator_LLM -- (item_name, date) --> get_item_stock;
    Orchestrator_LLM -- (date) --> get_full_inventory_summary;
    Orchestrator_LLM -- (keywords, limit) --> find_previous_quotes;
    Orchestrator_LLM -- (item_name, quantity) --> generate_price_quote;
    Orchestrator_LLM -- (item_name) --> get_item_unit_price;
    Orchestrator_LLM -- (item, type, qty, price, date) --> place_order;
    Orchestrator_LLM -- (item, qty, date) --> estimate_delivery_date;

    
    get_item_stock -- (Stock Info) --> Orchestrator_LLM;
    get_full_inventory_summary -- (Inventory Summary) --> Orchestrator_LLM;
    find_previous_quotes -- (Quote Details) --> Orchestrator_LLM;
    generate_price_quote -- (Calculated Quote) --> Orchestrator_LLM;
    get_item_unit_price -- (Unit Price) --> Orchestrator_LLM;
    place_order -- (Transaction Result) --> Orchestrator_LLM;
    estimate_delivery_date -- (Delivery Date) --> Orchestrator_LLM;

    
    Orchestrator_LLM -- (Final Response) --> User_Response[Customer Response];


    style User_Request fill:#f9f,stroke:#333,stroke-width:2px;
    style User_Response fill:#afa,stroke:#333,stroke-width:2px;
    style Orchestrator_LLM fill:#bbf,stroke:#333,stroke-width:2px;
    style Database fill:#ffd700,stroke:#333,stroke-width:2px;
    style Date_Calculation_Logic fill:#87ceeb,stroke:#333,stroke-width:2px;
    style Product_Catalog fill:#90ee90,stroke:#333,stroke-width:2px;
```

# 2. Smolagents framework/ tools

In [2]:
pip install smolagents

In [3]:
import os
import openai
from dotenv import load_dotenv
from google.colab import userdata
from typing import Dict, List, Union # Add this line

# Initialize the OpenAI client
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Import SmolAgents components
from smolagents.tools import tool
from smolagents import ToolCallingAgent as SmolAgent
from smolagents.models import OpenAIModel # Import OpenAIModel
from datetime import datetime, timedelta

# Set the OpenAI client for SmolAgents (removed Settings.set_llm, will pass to agents directly)

# Instantiate the OpenAIModel wrapper for the agents
llm_model = OpenAIModel(model_id="gpt-4o", client=openai_client) # Using gpt-4o as a capable default for tool calling

# Redefine existing tools as smolagents tools to avoid recursion and use original functions
@tool
def smol_get_item_stock(item_name: str, request_date: str = None) -> str:
    """
    Checks the current stock level for a given item.

    Args:
        item_name (str): The name of the item to check.
        request_date (str, optional): The date for which to check the stock in ISO format (YYYY-MM-DD).
                                       Defaults to the current date if not provided.

    Returns:
        str: A message indicating the stock level of the item or if it's not found.
    """
    if request_date is None:
        request_date = datetime.now().isoformat().split('T')[0]
    # Calls original get_item_stock
    return get_item_stock(item_name, request_date)

@tool
def smol_get_full_inventory_summary(request_date: str = None) -> str:
    """
    Retrieves a summary of all available inventory items and their current stock levels.

    Args:
        request_date (str, optional): The date for which to get the inventory summary in ISO format (YYYY-MM-DD).
                                       Defaults to the current date if not provided.

    Returns:
        str: A formatted string listing all available items and their stock, or a message if inventory is empty.
    """
    if request_date is None:
        request_date = datetime.now().isoformat().split('T')[0]
    # Calls original get_full_inventory_summary
    return get_full_inventory_summary(request_date)

@tool
def smol_find_previous_quotes(keywords: str, limit: int = 5) -> str:
    """
    Searches for previous quotes based on keywords and returns a summary.
    This tool is useful for understanding historical pricing, job types, and order details.

    Args:
        keywords (str): A comma-separated string of keywords to search for in past quotes.
                        Examples: 'conference, paper', 'party supplies', 'cardstock, wedding'
        limit (int, optional): The maximum number of previous quotes to return. Defaults to 5.

    Returns:
        str: A formatted string summarizing the found quotes, or a message if none are found.
    """
    search_terms = [term.strip() for term in keywords.split(',') if term.strip()]
    if not search_terms:
        return "Please provide at least one keyword to search for previous quotes."
    # Calls original find_previous_quotes
    return find_previous_quotes(search_terms, limit)

@tool
def smol_generate_price_quote(item_name: str, quantity: int) -> str:
    """
    Calculates a price quote for a given item and quantity, applying discounts.
    Discounts:
    - 0% for quantity <= 50
    - 5% for quantity > 50 and <= 200
    - 10% for quantity > 200
    Returns a string with the calculated quote.

    Args:
        item_name (str): The name of the item to generate a quote for.
        quantity (int): The quantity of the item for the quote.
    """
    # Calls original generate_price_quote
    return generate_price_quote(item_name, quantity)

@tool
def smol_get_item_unit_price(item_name: str) -> float:
    """
    Retrieves the unit price for a given item from the `paper_supplies` list, using fuzzy matching.

    Args:
        item_name (str): The name of the item to look up.

    Returns:
        float: The unit price of the item. Returns -1.0 if the item is not found or if the fuzzy match score is too low.
    """
    # Calls original get_item_unit_price
    return get_item_unit_price(item_name)

@tool
def smol_place_order(
    item_name: str,
    transaction_type: str,
    quantity: int,
    unit_price: float,
    request_date: str = None
) -> str:
    """
    Places an order by creating a new transaction record in the database.
    This tool can be used for 'stock_orders' (buying inventory) or 'sales' (selling to a customer).

    Args:
        item_name (str): The name of the item for the order.
        transaction_type (str): The type of transaction, either 'stock_orders' or 'sales'.
        quantity (int): The number of units to order/sell.
        unit_price: The price per unit of the item.
        request_date (str, optional): The date of the transaction in ISO format (YYYY-MM-DD).
                                      Defaults to the current date if not provided.

    Returns:
        str: A confirmation message or an error message if the order fails.
    """
    if request_date is None:
        request_date = datetime.now().isoformat().split('T')[0]
    # Calls original place_order
    return place_order(item_name, transaction_type, quantity, unit_price, request_date)

@tool
def smol_estimate_delivery_date(item_name: str, quantity: int, current_date: str = None) -> str:
    """
    Estimates the delivery date for a given quantity of an item from a supplier.

    Args:
        item_name (str): The name of the item to estimate delivery for.
        quantity (int): The quantity of the item to be ordered.
        current_date (str, optional): The current date in ISO format (YYYY-MM-DD). Defaults to today.

    Returns:
        str: A message indicating the estimated delivery date or an error if the date cannot be determined.
    """
    if current_date is None:
        current_date = datetime.now().isoformat().split('T')[0]
    # Calls original estimate_delivery_date
    return estimate_delivery_date(item_name, quantity, current_date)

@tool
def smol_get_cash_balance_tool(as_of_date: str = None) -> float:
    """
    Calculates the current cash balance as of a specified date.

    Args:
        as_of_date (str, optional): The cutoff date (inclusive) in ISO format (YYYY-MM-DD).
                                    Defaults to the current date if not provided.

    Returns:
        float: Net cash balance as of the given date. Returns 0.0 if no transactions exist or an error occurs.
    """
    if as_of_date is None:
        as_of_date = datetime.now().isoformat().split('T')[0]
    return get_cash_balance(as_of_date)

@tool
def smol_generate_financial_report_tool(as_of_date: str = None) -> Dict:
    """
    Generate a complete financial report for the company as of a specific date.

    This includes:-
    - Cash balance
    - Inventory valuation
    - Combined asset total
    - Itemized inventory breakdown
    - Top 5 best-selling products

    Args:
        as_of_date (str, optional): The date (inclusive) for which to generate the report in ISO format (YYYY-MM-DD).
                                    Defaults to the current date if not provided.

    Returns:
        Dict: A dictionary containing the financial report fields:
            - 'as_of_date': The date of the report
            - 'cash_balance': Total cash available
            - 'inventory_value': Total value of inventory
            - 'total_assets': Combined cash and inventory value
            - 'inventory_summary': List of items with stock and valuation details
            - 'top_selling_products': List of top 5 products by revenue
    """
    if as_of_date is None:
        as_of_date = datetime.now().isoformat().split('T')[0]
    return generate_financial_report(as_of_date)


# Define the SmolAgents
# Inventory Agent
inventory_agent = SmolAgent(
    name="InventoryAgent",
    description="Manages and tracks paper product stock levels, preventing stockouts.",
    model=llm_model,
    tools=[
        smol_get_item_stock,
        smol_get_full_inventory_summary
    ]
)

# Quoting Agent
quoting_agent = SmolAgent(
    name="QuotingAgent",
    description="Generates price quotes and accesses historical sales data.",
    model=llm_model,
    tools=[
        smol_find_previous_quotes,
        smol_generate_price_quote
    ]
)

# Ordering Agent
ordering_agent = SmolAgent(
    name="OrderingAgent",
    description="Handles procurement, sales, and delivery estimations.",
    model=llm_model,
    tools=[
        smol_get_item_unit_price,
        smol_place_order,
        smol_estimate_delivery_date
    ]
)

# Combine all tool functions into a single list for the OrchestrationAgent
all_tool_functions = [
    smol_get_item_stock,
    smol_get_full_inventory_summary,
    smol_find_previous_quotes,
    smol_generate_price_quote,
    smol_get_item_unit_price,
    smol_place_order,
    smol_estimate_delivery_date,
    smol_get_cash_balance_tool,
    smol_generate_financial_report_tool
]

# Orchestration Agent (main agent that uses other agents' tools)
orchestration_agent = SmolAgent(
    name="OrchestrationAgent",
    description="Coordinates the flow and delegates tasks to specialized agents based on user requests. **For purchase requests (e.g., 'I want to buy X units of Y'), the agent MUST perform the following steps to ensure financial records are updated:**\n1. **Accurately identify the `item_name` and `quantity` from the user's request.**\n2. **First, use the `smol_get_item_unit_price` tool with the extracted `item_name` to retrieve its `unit_price`.**\n3. **Then, use the `smol_place_order` tool with `item_name`, `transaction_type='sales'`, `quantity`, and the obtained `unit_price` to record the sale.**\nThis precise sequence is crucial for correctly updating the cash balance and inventory values. For other requests, the agent can check inventory (using `smol_get_item_stock` or `smol_get_full_inventory_summary`), provide quotes (using `smol_generate_price_quote` or `smol_find_previous_quotes`), and estimate delivery dates (using `smol_estimate_delivery_date`).",
    model=llm_model,
    tools=all_tool_functions
)

print("SmolAgents and tools defined.")

SmolAgents and tools defined.


# pydantic-ai

In [5]:
!pip install pydantic-ai

import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# Set the API key as an environment variable for pydantic-ai to pick it up
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

from pydantic_ai import Agent as PydanticAgent

inventory_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Inventory Agent. Your responsibility is to check stock levels, "
        "summarize inventory, and explain whether requested items are available."
    )
)

quoting_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Quoting Agent. Your responsibility is to search historical quotes, "
        "generate price quotes, and explain discount logic transparently."
    )
)

ordering_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Ordering Agent. Your responsibility is to retrieve unit prices, "
        "record confirmed sales or stock orders, and estimate delivery dates."
    )
)

orchestrator_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Orchestration Agent. Route customer requests to the correct worker agent. "
        "For confirmed purchase intent, ensure that the order is actually recorded through "
        "the Ordering Agent. Never tell the customer an order was placed unless the transaction "
        "tool confirms it."
    )
)

/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py:380: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


# project_starter.py

In [4]:
pip install -r requirements.txt

In [8]:
import pandas as pd
import numpy as np
import os
import time
import dotenv
import ast
from sqlalchemy.sql import text
from datetime import datetime, timedelta
from typing import Dict, List, Union
from sqlalchemy import create_engine, Engine
import openai # Import openai for the client object
import json # Import the json module
from thefuzz import fuzz, process

# Create an SQLite database
db_engine = create_engine("sqlite:///munder_difflin.db")

# List containing the different kinds of papers
paper_supplies = [
    # Paper Types (priced per sheet unless specified)
    {"item_name": "A4 paper",                         "category": "paper",        "unit_price": 0.05},
    {"item_name": "Letter-sized paper",              "category": "paper",        "unit_price": 0.06},
    {"item_name": "Cardstock",                        "category": "paper",        "unit_price": 0.15},
    {"item_name": "Colored paper",                    "category": "paper",        "unit_price": 0.10},
    {"item_name": "Glossy paper",                     "category": "paper",        "unit_price": 0.20},
    {"item_name": "Matte paper",                      "category": "paper",        "unit_price": 0.18},
    {"item_name": "Recycled paper",                   "category": "paper",        "unit_price": 0.08},
    {"item_name": "Eco-friendly paper",               "category": "paper",        "unit_price": 0.12},
    {"item_name": "Poster paper",                     "category": "paper",        "unit_price": 0.25},
    {"item_name": "Banner paper",                     "category": "paper",        "unit_price": 0.30},
    {"item_name": "Kraft paper",                      "category": "paper",        "unit_price": 0.10},
    {"item_name": "Construction paper",               "category": "paper",        "unit_price": 0.07},
    {"item_name": "Wrapping paper",                   "category": "paper",        "unit_price": 0.15},
    {"item_name": "Glitter paper",                    "category": "paper",        "unit_price": 0.22},
    {"item_name": "Decorative paper",                 "category": "paper",        "unit_price": 0.18},
    {"item_name": "Letterhead paper",                 "category": "paper",        "unit_price": 0.12},
    {"item_name": "Legal-size paper",                 "category": "paper",        "unit_price": 0.08},
    {"item_name": "Crepe paper",                      "category": "paper",        "unit_price": 0.05},
    {"item_name": "Photo paper",                      "category": "paper",        "unit_price": 0.25},
    {"item_name": "Uncoated paper",                   "category": "paper",        "unit_price": 0.06},
    {"item_name": "Butcher paper",                    "category": "paper",        "unit_price": 0.10},
    {"item_name": "Heavyweight paper",                "category": "paper",        "unit_price": 0.20},
    {"item_name": "Standard copy paper",              "category": "paper",        "unit_price": 0.04},
    {"item_name": "Bright-colored paper",             "category": "paper",        "unit_price": 0.12},
    {"item_name": "Patterned paper",                  "category": "paper",        "unit_price": 0.15},

    # Product Types (priced per unit)
    {"item_name": "Paper plates",                     "category": "product",      "unit_price": 0.10},  # per plate
    {"item_name": "Paper cups",                       "category": "product",      "unit_price": 0.08},  # per cup
    {"item_name": "Paper napkins",                    "category": "product",      "unit_price": 0.02},  # per napkin
    {"item_name": "Disposable cups",                  "category": "product",      "unit_price": 0.10},  # per cup
    {"item_name": "Table covers",                     "category": "product",      "unit_price": 1.50},  # per cover
    {"item_name": "Envelopes",                        "category": "product",      "unit_price": 0.05},  # per envelope
    {"item_name": "Sticky notes",                     "category": "product",      "unit_price": 0.03},  # per sheet
    {"item_name": "Notepads",                         "category": "product",      "unit_price": 2.00},  # per pad
    {"item_name": "Invitation cards",                 "category": "product",      "unit_price": 0.50},  # per card
    {"item_name": "Flyers",                           "category": "product",      "unit_price": 0.15},  # per flyer
    {"item_name": "Party streamers",                  "category": "product",      "unit_price": 0.05},  # per roll
    {"item_name": "Decorative adhesive tape (washi tape)", "category": "product", "unit_price": 0.20},  # per roll
    {"item_name": "Paper party bags",                 "category": "product",      "unit_price": 0.25},  # per bag
    {"item_name": "Name tags with lanyards",          "category": "product",      "unit_price": 0.75},  # per tag
    {"item_name": "Presentation folders",             "category": "product",      "unit_price": 0.50},

    # Large-format items (priced per unit)
    {"item_name": "Large poster paper (24x36 inches)", "category": "large_format", "unit_price": 1.00},
    {"item_name": "Rolls of banner paper (36-inch width)", "category": "large_format", "unit_price": 2.50},

    # Specialty papers
    {"item_name": "100 lb cover stock",               "category": "specialty",    "unit_price": 0.50},
    {"item_name": "80 lb text paper",                 "category": "specialty",    "unit_price": 0.40},
    {"item_name": "250 gsm cardstock",                "category": "specialty",    "unit_price": 0.30},
    {"item_name": "220 gsm poster paper",             "category": "specialty",    "unit_price": 0.35},
]

# Given below are some utility functions you can use to implement your multi-agent system

def generate_sample_inventory(paper_supplies: list, coverage: float = 0.4, seed: int = 137) -> pd.DataFrame:
    """
    Generate inventory for exactly a specified percentage of items from the full paper supply list.

    This function randomly selects exactly `coverage` × N items from the `paper_supplies` list,
    and assigns each selected item:
    - a random stock quantity between 200 and 800,
    - a minimum stock level between 50 and 150.

    The random seed ensures reproducibility of selection and stock levels.

    Args:
        paper_supplies (list): A list of dictionaries, each representing a paper item with
                               keys 'item_name', 'category', and 'unit_price'.
        coverage (float, optional): Fraction of items to include in the inventory (default is 0.4, or 40%).
        seed (int, optional): Random seed for reproducibility (default is 137).

    Returns:
        pd.DataFrame: A DataFrame with the selected items and assigned inventory values, including:
                      - item_name
                      - category
                      - unit_price
                      - current_stock
                      - min_stock_level
    """
    # Ensure reproducible random output
    np.random.seed(seed)

    # Calculate number of items to include based on coverage
    num_items = int(len(paper_supplies) * coverage)

    # Randomly select item indices without replacement
    selected_indices = np.random.choice(
        range(len(paper_supplies)),
        size=num_items,
        replace=False
    )

    # Extract selected items from paper_supplies list
    selected_items = [paper_supplies[i] for i in selected_indices]

    # Construct inventory records
    inventory = []
    for item in selected_items:
        inventory.append({
            "item_name": item["item_name"],
            "category": item["category"],
            "unit_price": item["unit_price"],
            "current_stock": np.random.randint(200, 800),  # Realistic stock range
            "min_stock_level": np.random.randint(50, 150)  # Reasonable threshold for reordering
        })

    # Return inventory as a pandas DataFrame
    return pd.DataFrame(inventory)

def init_database(db_engine: Engine, seed: int = 137) -> Engine:
    """
    Set up the Munder Difflin database with all required tables and initial records.

    This function performs the following tasks:
    - Creates the 'transactions' table for logging stock orders and sales
    - Loads customer inquiries from 'quote_requests.csv' into a 'quote_requests' table
    - Loads previous quotes from 'quotes.csv' into a 'quotes' table, extracting useful metadata
    - Generates a random subset of paper inventory using `generate_sample_inventory`
    - Inserts initial financial records including available cash and starting stock levels

    Args:
        db_engine (Engine): A SQLAlchemy engine connected to the SQLite database.
        seed (int, optional): A random seed used to control reproducibility of inventory stock levels.
                              Default is 137.

    Returns:
        Engine: The same SQLAlchemy engine, after initializing all necessary tables and records.

    Raises:
        Exception: If an error occurs during setup, the exception is printed and raised.
    """
    try:
        # ----------------------------
        # 1. Create an empty 'transactions' table schema
        # ----------------------------
        transactions_schema = pd.DataFrame({
            "id": [],
            "item_name": [],
            "transaction_type": [],  # 'stock_orders' or 'sales'
            "units": [],             # Quantity involved
            "price": [],             # Total price for the transaction
            "transaction_date": [],  # ISO-formatted date
        })
        transactions_schema.to_sql("transactions", db_engine, if_exists="replace", index=False)

        # Set a consistent starting date
        initial_date = datetime(2025, 1, 1).isoformat()

        # ----------------------------
        # 2. Load and initialize 'quote_requests' table
        # ----------------------------
        quote_requests_df = pd.read_csv("quote_requests.csv")
        quote_requests_df["id"] = range(1, len(quote_requests_df) + 1)
        quote_requests_df.to_sql("quote_requests", db_engine, if_exists="replace", index=False)

        # ----------------------------
        # 3. Load and transform 'quotes' table
        # ----------------------------
        quotes_df = pd.read_csv("quotes.csv")
        quotes_df["request_id"] = range(1, len(quotes_df) + 1)
        quotes_df["order_date"] = initial_date

        # Unpack metadata fields (job_type, order_size, event_type) if present
        if "request_metadata" in quotes_df.columns:
            quotes_df["request_metadata"] = quotes_df["request_metadata"].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            quotes_df["job_type"] = quotes_df["request_metadata"].apply(lambda x: x.get("job_type", ""))
            quotes_df["order_size"] = quotes_df["request_metadata"].apply(lambda x: x.get("order_size", ""))
            quotes_df["event_type"] = quotes_df["request_metadata"].apply(lambda x: x.get("event_type", ""))

        # Retain only relevant columns
        quotes_df = quotes_df[[
            "request_id",
            "total_amount",
            "quote_explanation",
            "order_date",
            "job_type",
            "order_size",
            "event_type"
        ]]
        quotes_df.to_sql("quotes", db_engine, if_exists="replace", index=False)

        # ----------------------------
        # 4. Generate inventory and seed stock
        # ----------------------------
        inventory_df = generate_sample_inventory(paper_supplies, seed=seed)

        # Seed initial transactions
        initial_transactions = []

        # Add a starting cash balance via a dummy sales transaction
        initial_transactions.append({
            "item_name": None,
            "transaction_type": "sales",
            "units": None,
            "price": 50000.0,
            "transaction_date": initial_date,
        })

        # Add one stock order transaction per inventory item
        for _, item in inventory_df.iterrows():
            initial_transactions.append({
                "item_name": item["item_name"],
                "transaction_type": "stock_orders",
                "units": item["current_stock"],
                "price": item["current_stock"] * item["unit_price"],
                "transaction_date": initial_date,
            })

        # Commit transactions to database
        pd.DataFrame(initial_transactions).to_sql("transactions", db_engine, if_exists="append", index=False)

        # Save the inventory reference table
        inventory_df.to_sql("inventory", db_engine, if_exists="replace", index=False)

        return db_engine

    except Exception as e:
        print(f"Error initializing database: {e}")
        raise

def create_transaction(
    item_name: str,
    transaction_type: str,
    quantity: int,
    price: float,
    date: Union[str, datetime],
) -> int:
    """
    This function records a transaction of type 'stock_orders' or 'sales' with a specified
    item name, quantity, total price, and transaction date into the 'transactions' table of the database.

    Args:
        item_name (str): The name of the item involved in the transaction.
        transaction_type (str): Either 'stock_orders' or 'sales'.
        quantity (int): Number of units involved in the transaction.
        price (float): Total price of the transaction.
        date (str or datetime): Date of the transaction in ISO 8601 format.

    Returns:
        int: The ID of the newly inserted transaction.

    Raises:
        ValueError: If `transaction_type` is not 'stock_orders' or 'sales'.
        Exception: For other database or execution errors.
    """
    try:
        # Convert datetime to ISO string if necessary
        date_str = date.isoformat() if isinstance(date, datetime) else date

        # Validate transaction type
        if transaction_type not in {"stock_orders", "sales"}:
            raise ValueError("Transaction type must be 'stock_orders' or 'sales'")

        # Prepare transaction record as a single-row DataFrame
        transaction = pd.DataFrame([
            {
                "item_name": item_name,
                "transaction_type": transaction_type,
                "units": quantity,
                "price": price,
                "transaction_date": date_str,
            }
        ])

        # Insert the record into the database
        transaction.to_sql("transactions", db_engine, if_exists="append", index=False)

        # Fetch and return the ID of the inserted row
        result = pd.read_sql("SELECT last_insert_rowid() as id", db_engine)
        return int(result.iloc[0]["id"])

    except Exception as e:
        print(f"Error creating transaction: {e}")
        raise

def get_all_inventory(as_of_date: str) -> Dict[str, int]:
    """
    Retrieve a snapshot of available inventory as of a specific date.

    This function calculates the net quantity of each item by summing
    all stock orders and subtracting all sales up to and including the given date.

    Only items with positive stock are included in the result.

    Args:
        as_of_date (str): ISO-formatted date string (YYYY-MM-DD) representing the inventory cutoff.

    Returns:
        Dict[str, int]: A dictionary mapping item names to their current stock levels.
    """
    # SQL query to compute stock levels per item as of the given date
    query = """
        SELECT
            item_name,
            SUM(CASE
                WHEN transaction_type = 'stock_orders' THEN units
                WHEN transaction_type = 'sales' THEN -units
                ELSE 0
            END) as stock
        FROM transactions
        WHERE item_name IS NOT NULL
        AND transaction_date <= :as_of_date
        GROUP BY item_name
        HAVING stock > 0
    """

    # Execute the query with the date parameter
    result = pd.read_sql(query, db_engine, params={"as_of_date": as_of_date})

    # Convert the result into a dictionary {item_name: stock}
    return dict(zip(result["item_name"], result["stock"]))

def get_stock_level(item_name: str, as_of_date: Union[str, datetime]) -> pd.DataFrame:
    """
    Retrieve the stock level of a specific item as of a given date.

    This function calculates the net stock by summing all 'stock_orders' and
    subtracting all 'sales' transactions for the specified item up to the given date.

    Args:
        item_name (str): The name of the item to look up.
        as_of_date (str or datetime): The cutoff date (inclusive) for calculating stock.

    Returns:
        pd.DataFrame: A single-row DataFrame with columns 'item_name' and 'current_stock'.
    """
    # Convert date to ISO string format if it's a datetime object
    if isinstance(as_of_date, datetime):
        as_of_date = as_of_date.isoformat()

    # SQL query to compute net stock level for the item
    stock_query = """
        SELECT
            item_name,
            COALESCE(SUM(CASE
                WHEN transaction_type = 'stock_orders' THEN units
                WHEN transaction_type = 'sales' THEN -units
                ELSE 0
            END), 0) AS current_stock
        FROM transactions
        WHERE item_name = :item_name
        AND transaction_date <= :as_of_date
    """

    # Execute query and return result as a DataFrame
    return pd.read_sql(
        stock_query,
        db_engine,
        params={"item_name": item_name, "as_of_date": as_of_date},
    )
# ================================================
def get_supplier_delivery_date(input_date_str: str, quantity: int) -> str:
    """
    Estimate supplier delivery date based on requested quantity and starting date.

    Lead-time rules:
    - <=10 units: same day
    - 11-100 units: 1 day
    - 101-1000 units: 4 days
    - >1000 units: 7 days
    """
    try:
        input_date_dt = datetime.fromisoformat(input_date_str.split("T")[0])
    except (ValueError, TypeError):
        input_date_dt = datetime.now()

    if quantity <= 10:
        days = 0
    elif quantity <= 100:
        days = 1
    elif quantity <= 1000:
        days = 4
    else:
        days = 7

    return (input_date_dt + timedelta(days=days)).strftime("%Y-%m-%d")

# ==================================================================

def get_cash_balance(as_of_date: Union[str, datetime]) -> float:
    """
    Calculate the current cash balance as of a specified date.

    The balance is computed by subtracting total stock purchase costs ('stock_orders')
    from total revenue ('sales') recorded in the transactions table up to the given date.

    Args:
        as_of_date (str or datetime): The cutoff date (inclusive) in ISO format or as a datetime object.

    Returns:
        float: Net cash balance as of the given date. Returns 0.0 if no transactions exist or an error occurs.
    """
    try:
        # Convert date to ISO format if it's a datetime object
        if isinstance(as_of_date, datetime):
            as_of_date = as_of_date.isoformat()

        # Query all transactions on or before the specified date
        transactions = pd.read_sql(
            "SELECT * FROM transactions WHERE transaction_date <= :as_of_date",
            db_engine,
            params={"as_of_date": as_of_date},
        )

        # Compute the difference between sales and stock purchases
        if not transactions.empty:
            total_sales = transactions.loc[transactions["transaction_type"] == "sales", "price"].sum()
            total_purchases = transactions.loc[transactions["transaction_type"] == "stock_orders", "price"].sum()
            return float(total_sales - total_purchases)

        return 0.0

    except Exception as e:
        print(f"Error getting cash balance: {e}")
        return 0.0


def generate_financial_report(as_of_date: Union[str, datetime]) -> Dict:
    """
    Generate a complete financial report for the company as of a specific date.

    This includes:-
    - Cash balance
    - Inventory valuation
    - Combined asset total
    - Itemized inventory breakdown
    - Top 5 best-selling products

    Args:
        as_of_date (str or datetime): The date (inclusive) for which to generate the report.

    Returns:
        Dict: A dictionary containing the financial report fields:
            - 'as_of_date': The date of the report
            - 'cash_balance': Total cash available
            - 'inventory_value': Total value of inventory
            - 'total_assets': Combined cash and inventory value
            - 'inventory_summary': List of items with stock and valuation details
            - 'top_selling_products': List of top 5 products by revenue
    """
    # Normalize date input
    if isinstance(as_of_date, datetime):
        as_of_date = as_of_date.isoformat()

    # Get current cash balance
    cash = get_cash_balance(as_of_date)

    # Get current inventory snapshot
    inventory_df = pd.read_sql("SELECT * FROM inventory", db_engine)
    inventory_value = 0.0
    inventory_summary = []

    # Compute total inventory value and summary by item
    for _, item in inventory_df.iterrows():
        stock_info = get_stock_level(item["item_name"], as_of_date)
        stock = stock_info["current_stock"].iloc[0]
        item_value = stock * item["unit_price"]
        inventory_value += item_value

        inventory_summary.append({
            "item_name": item["item_name"],
            "stock": stock,
            "unit_price": item["unit_price"],
            "value": item_value,
        })

    # Identify top-selling products by revenue
    top_sales_query = """
        SELECT item_name, SUM(units) as total_units, SUM(price) as total_revenue
        FROM transactions
        WHERE transaction_type = 'sales' AND transaction_date <= :date
        GROUP BY item_name
        ORDER BY total_revenue DESC
        LIMIT 5
    """
    top_sales = pd.read_sql(top_sales_query, db_engine, params={"date": as_of_date})
    top_selling_products = top_sales.to_dict(orient="records")

    return {
        "as_of_date": as_of_date,
        "cash_balance": cash,
        "inventory_value": inventory_value,
        "total_assets": cash + inventory_value,
        "inventory_summary": inventory_summary,
        "top_selling_products": top_selling_products,
    }


def search_quote_history(search_terms: List[str], limit: int = 5) -> List[Dict]:
    """
    Retrieve a list of historical quotes that match any of the provided search terms.

    The function searches both the original customer request (from `quote_requests`) and
    the explanation for the quote (from `quotes`) for each keyword. Results are sorted by
    most recent order date and limited by the `limit` parameter.

    Args:
        search_terms (List[str]): List of terms to match against customer requests and explanations.
        limit (int, optional): Maximum number of quote records to return. Default is 5.

    Returns:
        List[Dict]: A list of matching quotes, each represented as a dictionary with fields:
            - original_request
            - total_amount
            - quote_explanation
            - job_type
            - order_size
            - event_type
            - order_date
    """
    conditions = []
    params = {}

    # Build SQL WHERE clause using LIKE filters for each search term
    for i, term in enumerate(search_terms):
        param_name = f"term_{i}"
        conditions.append(
            f"(LOWER(qr.response) LIKE :{param_name} OR "
            f"LOWER(q.quote_explanation) LIKE :{param_name}))")
        params[param_name] = f"%{term.lower()}%"

    # Combine conditions; fallback to always-true if no terms provided
    where_clause = " AND ".join(conditions) if conditions else "1=1"

    # Final SQL query to join quotes with quote_requests
    query = f"""
        SELECT
            qr.response AS original_request,
            q.total_amount,
            q.quote_explanation,
            q.job_type,
            q.order_size,
            q.event_type,
            q.order_date
        FROM quotes q
        JOIN quote_requests qr ON q.request_id = qr.id
        WHERE {where_clause}
        ORDER BY q.order_date DESC
        LIMIT {limit}
    """

    # Execute parameterized query
    with db_engine.connect() as conn:
        result = conn.execute(text(query), params)
        return [dict(row._mapping) for row in result]

########################
########################
########################
# YOUR MULTI AGENT STARTS HERE
########################
########################
########################

########################
########################
########################
# YOUR MULTI AGENT STARTS HERE
########################
########################
########################

import os
import re
import time
from typing import Dict, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent as PydanticAgent

# ------------------------------------------------------------
# Environment setup
# ------------------------------------------------------------
# IMPORTANT:
# - For a .py submission, do NOT write !pip install inside this file.
# - Install dependencies separately:
#   pip install pydantic-ai python-dotenv thefuzz[speedup]

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    try:
        # Optional Colab fallback. Safe to ignore in normal .py execution.
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
        if OPENAI_API_KEY:
            os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    except Exception:
        pass

if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is missing. Add it to your .env file or Colab secrets."
    )


# ------------------------------------------------------------
# Structured request model
# ------------------------------------------------------------
class ParsedCustomerRequest(BaseModel):
    """
    Structured representation of a customer request.

    The orchestrator uses this model to reduce reliance on free-form LLM
    decisions for critical transaction actions.
    """
    intent: str = Field(
        default="general",
        description="One of: purchase, quote, stock_check, delivery, history, general."
    )
    item_name: Optional[str] = Field(default=None, description="Matched item name.")
    quantity: Optional[int] = Field(default=None, description="Requested quantity.")
    request_date: str = Field(default="", description="Request date in YYYY-MM-DD format.")
    customer_context: str = Field(default="", description="Useful business context from the request.")


# ------------------------------------------------------------
# Utility functions for deterministic orchestration
# ------------------------------------------------------------
def normalize_date_from_request(request_text: str) -> str:
    """
    Extract a YYYY-MM-DD date from the request text.
    If no date is found, return today's date.
    """
    match = re.search(r"\d{4}-\d{2}-\d{2}", request_text)
    if match:
        return match.group(0)
    return datetime.now().strftime("%Y-%m-%d")


def extract_quantity(request_text: str) -> Optional[int]:
    """
    Extract the most likely quantity from a customer request.
    Handles comma-separated numbers such as 1,000.
    """
    cleaned = request_text.replace(",", "")
    numbers = re.findall(r"\b\d+\b", cleaned)

    if not numbers:
        return None

    # Ignore year-like values from dates where possible.
    candidate_numbers = []
    for number in numbers:
        value = int(number)
        if value >= 1900 and value <= 2100:
            continue
        candidate_numbers.append(value)

    if not candidate_numbers:
        return None

    # Usually the largest non-date number is the order quantity.
    return max(candidate_numbers)


def match_item_name(request_text: str) -> Optional[str]:
    """
    Match a customer request to the closest item in paper_supplies.
    Uses fuzzy matching to tolerate wording differences.
    """
    item_names = [item["item_name"] for item in paper_supplies]
    best_match = process.extractOne(
        request_text,
        item_names,
        scorer=fuzz.token_set_ratio,
        score_cutoff=60
    )

    if best_match:
        return best_match[0]

    return None


def detect_intent(request_text: str) -> str:
    """
    Detect the business intent of the request using transparent rules.
    Deterministic routing is used for purchase/order requests so that
    database transactions are actually committed.
    """
    text = request_text.lower()

    purchase_keywords = [
        "place order",
        "proceed",
        "buy",
        "purchase",
        "order",
        "confirm",
        "we need",
        "i need",
        "send us",
        "ship",
        "deliver"
    ]

    quote_keywords = [
        "quote",
        "price",
        "pricing",
        "cost",
        "how much",
        "estimate"
    ]

    stock_keywords = [
        "stock",
        "available",
        "availability",
        "inventory",
        "do you have"
    ]

    delivery_keywords = [
        "delivery",
        "deliver",
        "lead time",
        "when can"
    ]

    history_keywords = [
        "previous quote",
        "past quote",
        "similar quote",
        "history"
    ]

    if any(keyword in text for keyword in purchase_keywords):
        return "purchase"

    if any(keyword in text for keyword in delivery_keywords):
        return "delivery"

    if any(keyword in text for keyword in stock_keywords):
        return "stock_check"

    if any(keyword in text for keyword in history_keywords):
        return "history"

    if any(keyword in text for keyword in quote_keywords):
        return "quote"

    return "general"


def extract_order_details(request_with_date: str) -> ParsedCustomerRequest:
    """
    Extract structured order/request details from free-form customer text.
    """
    intent = detect_intent(request_with_date)
    item_name = match_item_name(request_with_date)
    quantity = extract_quantity(request_with_date)
    request_date = normalize_date_from_request(request_with_date)

    return ParsedCustomerRequest(
        intent=intent,
        item_name=item_name,
        quantity=quantity,
        request_date=request_date,
        customer_context=request_with_date
    )


def count_transactions(as_of_date: str) -> int:
    """
    Count all transactions up to a given date.
    Used for evaluation transparency.
    """
    query = """
        SELECT COUNT(*) AS transaction_count
        FROM transactions
        WHERE transaction_date <= :as_of_date
    """
    result = pd.read_sql(query, db_engine, params={"as_of_date": as_of_date})
    return int(result.iloc[0]["transaction_count"])


# ------------------------------------------------------------
# Inventory Agent tools
# ------------------------------------------------------------
def get_item_stock(item_name: str, request_date: str = None) -> str:
    """
    Check the current stock level for a given item.

    Args:
        item_name: Name of the item to check.
        request_date: Stock-check date in YYYY-MM-DD format.

    Returns:
        Customer-facing stock availability message.
    """
    if request_date is None:
        request_date = datetime.now().strftime("%Y-%m-%d")

    matched_item = match_item_name(item_name) or item_name
    stock_df = get_stock_level(matched_item, request_date)
    current_stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0

    if current_stock > 0:
        return f"Current stock for '{matched_item}' as of {request_date}: {current_stock} units."

    return f"'{matched_item}' is currently out of stock or not found as of {request_date}."


def get_full_inventory_summary(request_date: str = None) -> str:
    """
    Return a summary of all available inventory items and their stock levels.

    Args:
        request_date: Inventory date in YYYY-MM-DD format.

    Returns:
        Formatted inventory summary.
    """
    if request_date is None:
        request_date = datetime.now().strftime("%Y-%m-%d")

    inventory = get_all_inventory(request_date)

    if not inventory:
        return f"No items currently in stock as of {request_date}."

    summary = f"Current inventory as of {request_date}:\n"
    for item, stock in inventory.items():
        summary += f"- {item}: {int(stock)} units\n"

    return summary


# ------------------------------------------------------------
# Quoting Agent tools
# ------------------------------------------------------------
def get_item_unit_price(item_name: str) -> float:
    """
    Retrieve unit price for an item using fuzzy matching.

    Args:
        item_name: Name of the requested item.

    Returns:
        Unit price if found, otherwise -1.0.
    """
    item_names = [item["item_name"] for item in paper_supplies]
    best_match = process.extractOne(
        item_name,
        item_names,
        scorer=fuzz.token_set_ratio,
        score_cutoff=70
    )

    if not best_match:
        return -1.0

    matched_item_name = best_match[0]

    for item in paper_supplies:
        if item["item_name"] == matched_item_name:
            return float(item["unit_price"])

    return -1.0


def find_previous_quotes(keywords: str, limit: int = 5) -> str:
    """
    Search previous quotes using keywords.

    Args:
        keywords: Comma-separated search terms.
        limit: Maximum number of previous quotes to return.

    Returns:
        Customer-safe summary of matching quote history.
    """
    search_terms = [term.strip() for term in keywords.split(",") if term.strip()]

    if not search_terms:
        return "Please provide at least one keyword to search for previous quotes."

    quotes = search_quote_history(search_terms, limit)

    if not quotes:
        return f"No previous quotes found matching the keywords: {keywords}."

    summary = f"Found {len(quotes)} previous quotes matching '{keywords}':\n"

    for i, quote in enumerate(quotes, start=1):
        summary += f"\n--- Quote {i} ---\n"
        summary += f"Request: {quote.get('original_request', 'N/A')}\n"
        summary += f"Amount: ${float(quote.get('total_amount', 0.0)):.2f}\n"
        summary += f"Explanation: {quote.get('quote_explanation', 'N/A')}\n"
        summary += f"Job Type: {quote.get('job_type', 'N/A')}\n"
        summary += f"Event Type: {quote.get('event_type', 'N/A')}\n"
        summary += f"Order Date: {quote.get('order_date', 'N/A')}\n"

    return summary


def generate_price_quote(
    item_name: str,
    quantity: int,
    customer_context: str = ""
) -> str:
    """
    Generate a transparent customer-facing price quote.

    Discount policy:
    - Below 100 units: 0%
    - 100 to 499 units: 5%
    - 500 to 999 units: 10%
    - 1000+ units: 15%

    Args:
        item_name: Requested item.
        quantity: Requested quantity.
        customer_context: Optional request context.

    Returns:
        Price quote with rationale.
    """
    if quantity is None or quantity <= 0:
        return "Unable to generate quote: quantity must be greater than zero."

    matched_item = match_item_name(item_name) or item_name
    unit_price = get_item_unit_price(matched_item)

    if unit_price < 0:
        return f"Unable to generate quote because '{item_name}' was not found in the product price list."

    if quantity >= 1000:
        discount_rate = 0.15
    elif quantity >= 500:
        discount_rate = 0.10
    elif quantity >= 100:
        discount_rate = 0.05
    else:
        discount_rate = 0.00

    subtotal = quantity * unit_price
    discount_amount = subtotal * discount_rate
    final_total = subtotal - discount_amount

    context_note = f" Context considered: {customer_context}" if customer_context else ""

    return (
        f"Quote for {quantity} units of {matched_item}:\n"
        f"- Unit price: ${unit_price:.2f}\n"
        f"- Subtotal: ${subtotal:.2f}\n"
        f"- Discount applied: {discount_rate:.0%} (${discount_amount:.2f})\n"
        f"- Final quoted total: ${final_total:.2f}\n"
        f"Rationale: The quote is based on the listed unit price and the quantity-based "
        f"discount policy.{context_note}"
    )


# ------------------------------------------------------------
# Ordering Agent tools
# ------------------------------------------------------------
def place_order(
    item_name: str,
    transaction_type: str,
    quantity: int,
    unit_price: float,
    request_date: str = None
) -> str:
    """
    Record a stock order or customer sale in the transaction database.

    For sales, the function checks available stock. If stock is insufficient
    but the item exists in the price list, it first records a stock order for
    the shortage, then records the customer sale. This allows the system to
    fulfill valid customer demand while still keeping transaction records clear.

    Args:
        item_name: Item being ordered or sold.
        transaction_type: Either 'stock_orders' or 'sales'.
        quantity: Number of units.
        unit_price: Unit price.
        request_date: Transaction date in YYYY-MM-DD format.

    Returns:
        Customer-safe confirmation or rejection message.
    """
    if request_date is None:
        request_date = datetime.now().strftime("%Y-%m-%d")

    if quantity is None or quantity <= 0:
        return "Order could not be processed because quantity must be greater than zero."

    if unit_price is None or unit_price < 0:
        return f"Order could not be processed because no valid price was found for '{item_name}'."

    if transaction_type not in {"stock_orders", "sales"}:
        return "Order could not be processed because the transaction type is invalid."

    matched_item = match_item_name(item_name) or item_name

    try:
        if transaction_type == "sales":
            stock_df = get_stock_level(matched_item, request_date)
            available_stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0

            if available_stock < quantity:
                shortage = quantity - available_stock

                # Restock first so the customer order can be fulfilled.
                restock_cost = shortage * unit_price
                restock_transaction_id = create_transaction(
                    item_name=matched_item,
                    transaction_type="stock_orders",
                    quantity=shortage,
                    price=restock_cost,
                    date=request_date
                )

                sale_total = quantity * unit_price
                sale_transaction_id = create_transaction(
                    item_name=matched_item,
                    transaction_type="sales",
                    quantity=quantity,
                    price=sale_total,
                    date=request_date
                )

                return (
                    f"Order confirmed. Transaction ID: {sale_transaction_id}. "
                    f"{quantity} units of {matched_item} were recorded as a sale "
                    f"on {request_date}. Total amount: ${sale_total:.2f}. "
                    f"Note: available stock was {available_stock} units, so "
                    f"{shortage} additional units were restocked first under "
                    f"stock transaction ID {restock_transaction_id}."
                )

        total_price = quantity * unit_price

        transaction_id = create_transaction(
            item_name=matched_item,
            transaction_type=transaction_type,
            quantity=quantity,
            price=total_price,
            date=request_date
        )

        transaction_label = "sale" if transaction_type == "sales" else "stock order"

        return (
            f"Order confirmed. Transaction ID: {transaction_id}. "
            f"{quantity} units of {matched_item} were recorded as a {transaction_label} "
            f"on {request_date}. Total amount: ${total_price:.2f}."
        )

    except Exception:
        return (
            "Order could not be processed due to an internal transaction error. "
            "Please try again or contact support."
        )


def estimate_delivery_date(item_name: str, quantity: int, current_date: str = None) -> str:
    """
    Estimate delivery date for a given item and quantity.

    Args:
        item_name: Requested item.
        quantity: Requested quantity.
        current_date: Starting date in YYYY-MM-DD format.

    Returns:
        Estimated delivery message.
    """
    if current_date is None:
        current_date = datetime.now().strftime("%Y-%m-%d")

    if quantity is None or quantity <= 0:
        return "Could not estimate delivery date because quantity must be greater than zero."

    matched_item = match_item_name(item_name) or item_name
    delivery_date = get_supplier_delivery_date(current_date, quantity)

    return (
        f"Estimated delivery date for {quantity} units of {matched_item}, "
        f"if ordered on {current_date}: {delivery_date}."
    )


# ------------------------------------------------------------
# Pydantic-AI agents with registered tools
# ------------------------------------------------------------
inventory_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Inventory Agent. Your responsibility is to check stock levels, "
        "summarize inventory, and explain whether requested items are available. "
        "Use only inventory-related tools."
    ),
    tools=[
        get_item_stock,
        get_full_inventory_summary
    ]
)

quoting_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Quoting Agent. Your responsibility is to search historical quotes, "
        "generate price quotes, and explain discount logic transparently. "
        "Do not confirm orders; only provide quote-related information."
    ),
    tools=[
        find_previous_quotes,
        generate_price_quote,
        get_item_unit_price
    ]
)

ordering_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Ordering Agent. Your responsibility is to retrieve prices, "
        "record confirmed sales or stock orders, and estimate delivery dates. "
        "Never claim an order is placed unless place_order confirms a transaction."
    ),
    tools=[
        get_item_unit_price,
        place_order,
        estimate_delivery_date
    ]
)

orchestrator_agent = PydanticAgent(
    model="openai:gpt-4o-mini",
    system_prompt=(
        "You are the Orchestration Agent for the Paper Company Sales Team. "
        "Classify customer intent and delegate to Inventory, Quoting, or Ordering. "
        "For confirmed purchase intent, ensure the transaction is actually recorded. "
        "Never tell the customer an order was placed unless the order tool confirms it."
    )
)


# ------------------------------------------------------------
# Multi-agent orchestration function
# ------------------------------------------------------------
def call_your_multi_agent_system(request_with_date: str) -> str:
    """
    Main multi-agent orchestration function.

    This function uses deterministic routing for critical transaction workflows
    and Pydantic-AI agents for agent-role alignment and non-critical responses.

    Args:
        request_with_date: Customer request with appended request date.

    Returns:
        Customer-facing response.
    """
    parsed = extract_order_details(request_with_date)

    # ----------------------------
    # Purchase intent: deterministic transaction route
    # ----------------------------
    if parsed.intent == "purchase":
        if not parsed.item_name:
            return "Order could not be processed because the requested item could not be identified."

        if not parsed.quantity:
            return "Order could not be processed because the requested quantity could not be identified."

        unit_price = get_item_unit_price(parsed.item_name)

        if unit_price < 0:
            return f"Unable to process the order because '{parsed.item_name}' was not found in the price list."

        order_result = place_order(
            item_name=parsed.item_name,
            transaction_type="sales",
            quantity=parsed.quantity,
            unit_price=unit_price,
            request_date=parsed.request_date
        )

        # Include delivery estimate only if the order was actually confirmed.
        if order_result.startswith("Order confirmed"):
            delivery_result = estimate_delivery_date(
                item_name=parsed.item_name,
                quantity=parsed.quantity,
                current_date=parsed.request_date
            )
            return f"{order_result}\n{delivery_result}"

        return order_result

    # ----------------------------
    # Quote intent
    # ----------------------------
    if parsed.intent == "quote":
        if parsed.item_name and parsed.quantity:
            return generate_price_quote(
                item_name=parsed.item_name,
                quantity=parsed.quantity,
                customer_context=parsed.customer_context
            )

        result = quoting_agent.run_sync(request_with_date)
        return getattr(result, "output", str(result))

    # ----------------------------
    # Stock inquiry
    # ----------------------------
    if parsed.intent == "stock_check":
        if parsed.item_name:
            return get_item_stock(parsed.item_name, parsed.request_date)

        result = inventory_agent.run_sync(request_with_date)
        return getattr(result, "output", str(result))

    # ----------------------------
    # Delivery inquiry
    # ----------------------------
    if parsed.intent == "delivery":
        if parsed.item_name and parsed.quantity:
            return estimate_delivery_date(
                item_name=parsed.item_name,
                quantity=parsed.quantity,
                current_date=parsed.request_date
            )

        result = ordering_agent.run_sync(request_with_date)
        return getattr(result, "output", str(result))

    # ----------------------------
    # Historical quote inquiry
    # ----------------------------
    if parsed.intent == "history":
        keywords = parsed.item_name or request_with_date
        return find_previous_quotes(keywords=keywords, limit=5)

    # ----------------------------
    # General fallback through orchestrator
    # ----------------------------
    result = orchestrator_agent.run_sync(request_with_date)
    return getattr(result, "output", str(result))


# ------------------------------------------------------------
# Evaluation runner
# ------------------------------------------------------------
def run_test_scenarios():
    """
    Run all sample customer requests and save results to test_results.csv.

    The output includes transaction and financial-change fields required for
    transparent evaluation.
    """
    print("Initializing Database...")
    init_database(db_engine)

    try:
        quote_requests_sample = pd.read_csv("quote_requests_sample.csv")
        quote_requests_sample["request_date"] = pd.to_datetime(
            quote_requests_sample["request_date"],
            format="%m/%d/%y",
            errors="coerce"
        )
        quote_requests_sample.dropna(subset=["request_date"], inplace=True)
        quote_requests_sample = quote_requests_sample.sort_values("request_date")

    except Exception as e:
        print(f"FATAL: Error loading test data: {e}")
        return []

    initial_date = quote_requests_sample["request_date"].min().strftime("%Y-%m-%d")
    initial_report = generate_financial_report(initial_date)

    current_cash = initial_report["cash_balance"]
    current_inventory = initial_report["inventory_value"]

    results = []

    for idx, row in quote_requests_sample.iterrows():
        request_id = idx + 1
        request_date = row["request_date"].strftime("%Y-%m-%d")
        request_with_date = f"{row['request']} (Date of request: {request_date})"

        before_report = generate_financial_report(request_date)
        before_cash = before_report["cash_balance"]
        before_inventory = before_report["inventory_value"]
        before_transaction_count = count_transactions(request_date)

        print(f"\n=== Request {request_id} ===")
        print(f"Context: {row.get('job', 'N/A')} organizing {row.get('event', 'N/A')}")
        print(f"Request Date: {request_date}")
        print(f"Cash Balance Before: ${before_cash:.2f}")
        print(f"Inventory Value Before: ${before_inventory:.2f}")

        response = call_your_multi_agent_system(request_with_date)

        after_report = generate_financial_report(request_date)
        after_cash = after_report["cash_balance"]
        after_inventory = after_report["inventory_value"]
        after_transaction_count = count_transactions(request_date)

        cash_changed = after_cash != before_cash
        inventory_changed = after_inventory != before_inventory
        transaction_recorded = after_transaction_count > before_transaction_count
        fulfilled = response.startswith("Order confirmed")

        print(f"Response: {response}")
        print(f"Updated Cash: ${after_cash:.2f}")
        print(f"Updated Inventory: ${after_inventory:.2f}")
        print(f"Transaction Recorded: {transaction_recorded}")

        results.append(
            {
                "request_id": request_id,
                "request_date": request_date,
                "customer_request": row["request"],
                "cash_balance": after_cash,
                "inventory_value": after_inventory,
                "cash_changed": cash_changed,
                "inventory_changed": inventory_changed,
                "transaction_recorded": transaction_recorded,
                "fulfilled": fulfilled,
                "response": response,
            }
        )

        current_cash = after_cash
        current_inventory = after_inventory

        time.sleep(1)

    final_date = quote_requests_sample["request_date"].max().strftime("%Y-%m-%d")
    final_report = generate_financial_report(final_date)

    print("\n===== FINAL FINANCIAL REPORT =====")
    print(f"Final Cash: ${final_report['cash_balance']:.2f}")
    print(f"Final Inventory: ${final_report['inventory_value']:.2f}")
    print(f"Final Total Assets: ${final_report['total_assets']:.2f}")

    results_df = pd.DataFrame(results)
    results_df.to_csv("test_results.csv", index=False)

    print("\n===== EVALUATION SUMMARY =====")
    print(f"Total Requests: {len(results_df)}")
    print(f"Fulfilled Orders: {int(results_df['fulfilled'].sum())}")
    print(f"Cash Changes: {int(results_df['cash_changed'].sum())}")
    print(f"Inventory Changes: {int(results_df['inventory_changed'].sum())}")
    print(f"Transactions Recorded: {int(results_df['transaction_recorded'].sum())}")
    print("Saved results to test_results.csv")

    return results


if __name__ == "__main__":
    results = run_test_scenarios()


/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py:380: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


Initializing Database...

=== Request 1 ===
Context: office manager organizing ceremony
Request Date: 2025-04-01
Cash Balance Before: $45059.70
Inventory Value Before: $4940.30
Response: Order confirmed. Transaction ID: 20. 200 units of A4 paper were recorded as a sale on 2025-04-01. Total amount: $10.00.
Estimated delivery date for 200 units of A4 paper, if ordered on 2025-04-01: 2025-04-05.
Updated Cash: $45069.70
Updated Inventory: $4930.30
Transaction Recorded: True

=== Request 2 ===
Context: hotel manager organizing parade
Request Date: 2025-04-03
Cash Balance Before: $45069.70
Inventory Value Before: $4930.30
Response: Order confirmed. Transaction ID: 22. 500 units of Poster paper were recorded as a sale on 2025-04-03. Total amount: $125.00. Note: available stock was 0 units, so 500 additional units were restocked first under stock transaction ID 21.
Estimated delivery date for 500 units of Poster paper, if ordered on 2025-04-03: 2025-04-07.
Updated Cash: $45069.70
Updated Inven